In [1]:
!pip install -q gradio

In [2]:
from google.colab import files
uploaded = files.upload()
weights_filename = list(uploaded.keys())[0]

Saving MobileNetV2_best.weights.h5 to MobileNetV2_best.weights.h5


In [3]:
import h5py

with h5py.File(weights_filename, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Group) and len(obj.keys()) > 0:
            print(name)
    f.visititems(show)

layers
layers/dense
layers/dense/vars
layers/dense_1
layers/dense_1/vars
layers/dropout
layers/functional
layers/functional/layers
layers/functional/layers/add
layers/functional/layers/add_1
layers/functional/layers/add_2
layers/functional/layers/add_3
layers/functional/layers/add_4
layers/functional/layers/add_5
layers/functional/layers/add_6
layers/functional/layers/add_7
layers/functional/layers/add_8
layers/functional/layers/add_9
layers/functional/layers/batch_normalization
layers/functional/layers/batch_normalization/vars
layers/functional/layers/batch_normalization_1
layers/functional/layers/batch_normalization_1/vars
layers/functional/layers/batch_normalization_10
layers/functional/layers/batch_normalization_10/vars
layers/functional/layers/batch_normalization_11
layers/functional/layers/batch_normalization_11/vars
layers/functional/layers/batch_normalization_12
layers/functional/layers/batch_normalization_12/vars
layers/functional/layers/batch_normalization_13
layers/functiona

In [7]:
import tensorflow as tf
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2

IMG_SIZE = (224, 224)
CLASS_NAMES = ['Calculus', 'Data caries', 'Gingivitis', 'Mouth Ulcer', 'hypodontia']
NUM_CLASSES = len(CLASS_NAMES)

DENSE1_NAME = "dense"
DENSE2_NAME = "dense_1"
NUM_CLASSES = 5   # replace with the real name from Cell 3

base_model = MobileNetV2(include_top=False, weights=None, input_shape=(*IMG_SIZE, 3))
inputs = Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu', name=DENSE1_NAME)(x)
x = Dropout(0.4)(x)
outputs = Dense(NUM_CLASSES, activation='softmax', name=DENSE2_NAME)(x)
model = Model(inputs, outputs)
model.load_weights(weights_filename)
print("Weights loaded successfully")

Weights loaded successfully


In [8]:
import gradio as gr
import numpy as np
from PIL import Image

def predict(image: Image.Image):
    image = image.convert("RGB").resize(IMG_SIZE)
    arr = tf.keras.utils.img_to_array(image) / 255.0
    arr = np.expand_dims(arr, axis=0)
    preds = model.predict(arr)[0]
    return {CLASS_NAMES[i]: float(preds[i]) for i in range(NUM_CLASSES)}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload oral image"),
    outputs=gr.Label(num_top_classes=NUM_CLASSES, label="Prediction"),
    title="🦷 OralVision - Oral Disease Detector",
    description="Upload a mouth/teeth photo and the model will predict the most likely oral condition."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ecd5556ea98b810e1a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
